# Intruction
Following [`preprocessing.ipynb`](/notebooks/preprocessing.ipynb), we can now begin with testing our
datasets on different models.

Similarly to our *exploratory data analysis* portion, we will focus on a curated set of agencies to
base our initial model selection. We will focus on the **NYPD**, **DOT**, and **TLC**.

We are again, attempting to determine **resolution time**. This is a regression task. As such, we
will evaluate our models based on **Mean Average Error**, during *EDA* we discovered that there do 
exist many outliers for all three agencies. *MAE* gives us a more robust statistic, and is less
sensitive to outliers than other statistics, like *RMSE*.

# Setup
Here, we will establish our three agencies, as well as create our `X` and `y` features. Since each 
agency contains different magnitudes and fields, we will implement our previously created 
preprocessing pipeline *per-agency*.

In [179]:
# imports
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# grab data
df = pd.read_csv(
    '../data/nyc311_2025.csv', 
    index_col='id', 
    dtype={'zipcode': 'category'},
    parse_dates=['created', 'closed']
)

In [180]:
df.info()

<class 'pandas.DataFrame'>
Index: 3475290 entries, 67351762 to 63577994
Data columns (total 11 columns):
 #   Column           Dtype         
---  ------           -----         
 0   created          datetime64[us]
 1   closed           datetime64[us]
 2   agency_name      str           
 3   problem          str           
 4   detail           str           
 5   borough          str           
 6   lat              float64       
 7   long             float64       
 8   method           str           
 9   zipcode          category      
 10  resolution_time  float64       
dtypes: category(1), datetime64[us](2), float64(3), str(5)
memory usage: 298.3 MB


In [181]:
# initialize agencies


nypd = df[df.agency_name == 'New York City Police Department']
dot = df[df.agency_name == 'Department of Transportation']
tlc = df[df.agency_name == 'Taxi and Limousine Commission']

pd.concat([nypd.head(2), dot.head(2), tlc.head(2)])

,created,closed,agency_name,problem,detail,borough,lat,long,method,zipcode,resolution_time
id,,,,,,,,,,,
67351762,2025-12-31 23:59:28,2026-01-01 00:40:32,New York City Police Department,Noise - Residential,Loud Music/Party,MANHATTAN,40.792141,-73.950097,MOBILE,10029,0.684
67344624,2025-12-31 23:59:23,2026-01-01 01:03:42,New York City Police Department,Noise - Residential,Loud Music/Party,MANHATTAN,40.825137,-73.949447,ONLINE,10031,1.072
67344470,2025-12-31 23:57:00,2026-01-09 01:53:00,Department of Transportation,Street Light Condition,Street Light Out,BRONX,40.871462,-73.830537,UNKNOWN,10475,193.933
67348068,2025-12-31 23:57:00,2026-01-07 09:09:00,Department of Transportation,Street Light Condition,Street Light Out,BRONX,40.861213,-73.825111,UNKNOWN,10475,153.200
67345192,2025-12-31 23:58:25,2026-01-02 11:55:46,Taxi and Limousine Commission,Lost Property,Bag/Wallet,MANHATTAN,40.738077,-73.992123,PHONE,10011,35.956
67348740,2025-12-31 23:20:23,2026-01-02 13:27:51,Taxi and Limousine Commission,Lost Property,Bag/Wallet,QUEENS,40.648320,-73.788281,ONLINE,11430,38.124


In [182]:
agencies = {
    'nypd': {
        'df': nypd
    },
    'dot': {
        'df': dot
    },
    'tlc': {
        'df': tlc
    }
}


## Pipeline import

In [183]:
import cloudpickle as cp

preprocessing_pipeline = None
with open('../models/preprocessing_pipeline.pkl', 'rb') as f:
    preprocessing_pipeline = cp.load(f)

## Train-test split

In [184]:
from sklearn.model_selection import train_test_split

def populate_train_test(agencies: dict):
    '''
    Will give each agency in the agency dict proper X_train, X_test, y_train, y_test splits usign
    `sklearn.model_selection.train_test_split()`
    '''
    
    X = ['created', 'closed', 'detail', 'borough', 'method', 'problem', 'zipcode'] # list of all used features
    y = 'resolution_time'
    for a in agencies.keys():
        curr_df = agencies[a]['df']
        X_train, X_test, y_train, y_test = train_test_split(curr_df[X], curr_df[y], random_state=42, train_size=0.8)
        # populate
        agencies[a]['X_train'] = X_train
        agencies[a]['X_test'] = X_test
        agencies[a]['y_train'] = y_train
        agencies[a]['y_test'] = y_test

In [185]:
populate_train_test(agencies)

In [186]:
for k in agencies.keys():
    print(k, len(agencies[k]['X_train']), len(agencies[k]['X_test']))

nypd 1361864 340466
dot 138912 34729
tlc 22430 5608


# Model Selection
Next, we will begin model selection.

Our process begins with simply creating a pipeline which contains both our `preprocessing_pipeline`
along with the model of interest. 

As stated previouly, we are going to evaluate our models using
**MAE**. Along with using *cross-validation*, with `cv=5`.

We are going to evaluate the following models:
- asdf
-  adsf
- asdf

In [192]:
# model testing function 
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_validate
from sklearn.metrics import r2_score
import warnings

def test_model(agencies: dict, model: any, model_name: str, verbose: int=0):
    for a in agencies.keys():
        pipe = Pipeline([
            ('preprocessing', preprocessing_pipeline),
            (model_name, model)
        ])

        if verbose > 0:
              print(f'''performing cross validation
        model: {model_name}
        agency: {a}''')

        # we will be warned many times when there is an infrequent category column used. this is intended 
        # behavior for rare `problem` or `detail` columns which are not caught in the initial training.
        results = None
        with warnings.catch_warnings():
                warnings.filterwarnings('ignore', category=UserWarning, message='.*infrequent.*')
                # use cross validation
                results = cross_validate(
                        pipe,
                        agencies[a]['X_train'],
                        agencies[a]['y_train'],
                        cv=5,
                        scoring='neg_mean_absolute_error',
                        n_jobs=4,
                        verbose=verbose,
                        return_estimator=True
                )

        if verbose > 0: 
                print(f'''cross validation results....
        cv_scores (MAE).......................{results['test_score']}
        mean MAE..............................{np.mean(results['test_score'])}''')

        best_model = results['estimator'][np.argmax(results['test_score'])] # best model

        # find R^2 of best model
        y_pred = best_model.predict(agencies[a]['X_train'])
        r2 = r2_score(agencies[a]['y_train'], y_pred)

        agencies[a][f'{model_name}_cv_scores'] = results['test_score']
        agencies[a][f'{model_name}_r2'] = r2

def print_scores(agencies, model_name):
        print(f'All scores for {model_name}:')
        for a in agencies.keys():
                res = agencies[a][f'{model_name}_cv_scores']
                MAE_mean = -np.mean(res)
                MAE_normalized_mean = MAE_mean / (agencies[a]['y_train'].max() - agencies[a]['y_train'].min())
                r2 = agencies[a][f'{model_name}_r2']

                print(f'''{model_name}:
        CV MAE:                 {res}
        MAE MEAN:               {MAE_mean}
        MAE NORMALIZED MEAN:    {MAE_normalized_mean}
        R^2:                    {r2}''')

In [193]:
from sklearn.linear_model import Ridge

test_model(agencies, Ridge(), 'Ridge')
print_scores(agencies, 'Ridge')

/Users/luisjaco/Desktop/development/project_folders/vsCode/nyc-311/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as the infrequent category.
  warnings.warn(msg, UserWarning)
/Users/luisjaco/Desktop/development/project_folders/vsCode/nyc-311/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as the infrequent category.
  warnings.warn(msg, UserWarning)
/Users/luisjaco/Desktop/development/project_folders/vsCode/nyc-311/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as the infrequent category.
  warnings.warn(msg, UserWarning)


All scores for Ridge:
Ridge:
        CV MAE:                 [-2.65554835 -2.62683828 -2.63743273 -2.60726351 -2.63771144]
        MAE MEAN:               2.6329588651619287
        MAE NORMALIZED MEAN:    0.0011636188914741284
        R^2:                    0.05083113212356949
Ridge:
        CV MAE:                 [-250.87652389 -253.13485512 -260.57640027 -262.65007655 -255.19568909]
        MAE MEAN:               256.4867089837914
        MAE NORMALIZED MEAN:    0.01884703716084626
        R^2:                    0.14802494605264804
Ridge:
        CV MAE:                 [-1304.69219274 -1286.66923338 -1312.98073708 -1297.15819297
 -1264.45346653]
        MAE MEAN:               1293.1907645403176
        MAE NORMALIZED MEAN:    0.10478016867526338
        R^2:                    0.34795327458690906


In [ ]:
from sklearn.linear_model import ElasticNet

test_model(agencies, ElasticNet(), 'ElasticNet')
print_scores(agencies, 'ElasticNet')

/Users/luisjaco/Desktop/development/project_folders/vsCode/nyc-311/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as the infrequent category.
  warnings.warn(msg, UserWarning)
/Users/luisjaco/Desktop/development/project_folders/vsCode/nyc-311/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as the infrequent category.
  warnings.warn(msg, UserWarning)
/Users/luisjaco/Desktop/development/project_folders/vsCode/nyc-311/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as the infrequent category.
  warnings.warn(msg, UserWarning)
/Users/luisjaco/Desktop/development/project_folders/

All scores for ElasticNet:
ElasticNet:
        CV MAE:                 [-2.71914807 -2.68695794 -2.69613064 -2.66336932 -2.70068555]
        MAE MEAN:               2.6932583026926173
        MAE NORMALIZED MEAN:    0.0011902678321713688
        R^2:                    0.030327739884828397
ElasticNet:
        CV MAE:                 [-277.84054701 -283.16511641 -286.9019641  -287.67906467 -283.06493478]
        MAE MEAN:               283.73032539571517
        MAE NORMALIZED MEAN:    0.020848939921990176
        R^2:                    0.05151424708552377
ElasticNet:
        CV MAE:                 [-1528.94819954 -1501.6206037  -1532.74349166 -1528.70335464
 -1489.09728156]
        MAE MEAN:               1516.222586220832
        MAE NORMALIZED MEAN:    0.12285121630135937
        R^2:                    0.2350611368248966


/Users/luisjaco/Desktop/development/project_folders/vsCode/nyc-311/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as the infrequent category.
  warnings.warn(msg, UserWarning)
/Users/luisjaco/Desktop/development/project_folders/vsCode/nyc-311/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as the infrequent category.
  warnings.warn(msg, UserWarning)
/Users/luisjaco/Desktop/development/project_folders/vsCode/nyc-311/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as the infrequent category.
  warnings.warn(msg, UserWarning)


In [ ]:
from sklearn.svm import LinearSVR

test_model(agencies, LinearSVR(), 'LinearSVR')
print_scores(agencies, 'LinearSVR')

/Users/luisjaco/Desktop/development/project_folders/vsCode/nyc-311/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as the infrequent category.
  warnings.warn(msg, UserWarning)
/Users/luisjaco/Desktop/development/project_folders/vsCode/nyc-311/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as the infrequent category.
  warnings.warn(msg, UserWarning)


KeyboardInterrupt: 

In [ ]:
# xgboost
import xgboost

test_model(agencies, xgboost.XGBRegressor(), 'XGBoost')
print_scores(agencies, 'XGBoost')

/Users/luisjaco/Desktop/development/project_folders/vsCode/nyc-311/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as the infrequent category.
  warnings.warn(msg, UserWarning)
/Users/luisjaco/Desktop/development/project_folders/vsCode/nyc-311/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as the infrequent category.
  warnings.warn(msg, UserWarning)
/Users/luisjaco/Desktop/development/project_folders/vsCode/nyc-311/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as the infrequent category.
  warnings.warn(msg, UserWarning)
/Users/luisjaco/Desktop/development/project_folders/

All scores for XGBoost:
NYPD:   
        results:    [-2.28027567 -2.26627107 -2.26519409 -2.22911376 -2.25790059]       
        mean:       2.2597510370258247
        R^2:        0.24018748129400969
TLC:    
        results:    [-1196.80168317 -1188.12891834 -1223.14052178 -1192.17962142
 -1189.17071831]
        mean:       1197.8842926033085
        R^2:        0.5032754449180739
DOT:    
        results:    [-230.90312034 -233.08244646 -243.7229425  -242.47131941 -237.22561623]
        mean:       237.48108898706587
        R^2:        0.26712985938135214



/Users/luisjaco/Desktop/development/project_folders/vsCode/nyc-311/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as the infrequent category.
  warnings.warn(msg, UserWarning)


In [ ]:
from sklearn.tree import DecisionTreeRegressor

test_model(agencies, DecisionTreeRegressor(criterion='absolute_error', max_depth=2), 'DT')
print_scores(agencies, 'DT')

/Users/luisjaco/Desktop/development/project_folders/vsCode/nyc-311/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as the infrequent category.
  warnings.warn(msg, UserWarning)
/Users/luisjaco/Desktop/development/project_folders/vsCode/nyc-311/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as the infrequent category.
  warnings.warn(msg, UserWarning)
/Users/luisjaco/Desktop/development/project_folders/vsCode/nyc-311/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as the infrequent category.
  warnings.warn(msg, UserWarning)
/Users/luisjaco/Desktop/development/project_folders/

KeyboardInterrupt: 